# NetFlow v2 Two-Stage CatBoost

1단계는 보수적인 `Benign vs Attack` 분류기로 오탐을 줄이고, 2단계는 공격으로 판정된 플로우의 유형만 분류한다.

In [1]:
from pathlib import Path
import json

import numpy as np
from catboost import CatBoostClassifier
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_curve
from sklearn.utils.class_weight import compute_class_weight

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'training').exists():
    PROJECT_ROOT = Path.cwd().parents[1]

PROCESSED_DIR = PROJECT_ROOT / 'training/data/processed_netflow_v2_twostage'
MODEL_DIR = PROJECT_ROOT / 'training/artifacts/models'
BINARY_MODEL_NAME = 'best_model_netflow_v2_binary_catboost.cbm'
ATTACK_MODEL_NAME = 'best_model_netflow_v2_attack_catboost.cbm'
TARGET_MAX_FPR = 0.01
MIN_ATTACK_RECALL = 0.80
RANDOM_STATE = 42
MODEL_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR

PosixPath('/Users/minseok/Desktop/codex/프로젝트/DeepFence/training/data/processed_netflow_v2_twostage')

In [2]:
x_train = np.load(PROCESSED_DIR / 'X_train.npy')
x_val = np.load(PROCESSED_DIR / 'X_val.npy')
x_test = np.load(PROCESSED_DIR / 'X_test.npy')
y_binary_train = np.load(PROCESSED_DIR / 'y_binary_train.npy')
y_binary_val = np.load(PROCESSED_DIR / 'y_binary_val.npy')
y_binary_test = np.load(PROCESSED_DIR / 'y_binary_test.npy')
x_attack_train = np.load(PROCESSED_DIR / 'X_attack_train.npy')
x_attack_val = np.load(PROCESSED_DIR / 'X_attack_val.npy')
x_attack_test = np.load(PROCESSED_DIR / 'X_attack_test.npy')
y_attack_train = np.load(PROCESSED_DIR / 'y_attack_train.npy')
y_attack_val = np.load(PROCESSED_DIR / 'y_attack_val.npy')
y_attack_test = np.load(PROCESSED_DIR / 'y_attack_test.npy')

attack_label_mapping = json.loads((PROCESSED_DIR / 'attack_label_mapping.json').read_text(encoding='utf-8'))
attack_idx_to_label = {index: label for label, index in attack_label_mapping.items()}
attack_names = [attack_idx_to_label[index] for index in sorted(attack_idx_to_label)]
x_train.shape, x_attack_train.shape, attack_names

((1174453, 17),
 (474453, 17),
 ['Bot', 'Brute Force', 'DDoS', 'DoS', 'Infiltration', 'SQL Injection'])

In [3]:
binary_model = CatBoostClassifier(
    loss_function='Logloss',
    eval_metric='F1',
    iterations=800,
    learning_rate=0.06,
    depth=8,
    class_weights=[1.8, 1.0],  # Benign 오탐 비용을 더 크게 둔다.
    random_seed=RANDOM_STATE,
    od_type='Iter',
    od_wait=50,
    verbose=50,
)
binary_model.fit(x_train, y_binary_train, eval_set=(x_val, y_binary_val), use_best_model=True)

0:	learn: 0.9505159	test: 0.9506644	best: 0.9506644 (0)	total: 125ms	remaining: 1m 40s
50:	learn: 0.9557770	test: 0.9558904	best: 0.9558939 (48)	total: 2.86s	remaining: 42s
100:	learn: 0.9563985	test: 0.9564916	best: 0.9564916 (100)	total: 5.57s	remaining: 38.6s
150:	learn: 0.9568519	test: 0.9569231	best: 0.9569231 (150)	total: 8.09s	remaining: 34.7s
200:	learn: 0.9572087	test: 0.9573194	best: 0.9573283 (199)	total: 10.7s	remaining: 31.8s
250:	learn: 0.9573947	test: 0.9574457	best: 0.9574511 (248)	total: 13.2s	remaining: 28.9s
300:	learn: 0.9575116	test: 0.9575722	best: 0.9575722 (298)	total: 15.7s	remaining: 26s
350:	learn: 0.9576298	test: 0.9576646	best: 0.9576646 (347)	total: 18.2s	remaining: 23.3s
400:	learn: 0.9576998	test: 0.9577534	best: 0.9577534 (395)	total: 20.8s	remaining: 20.7s
450:	learn: 0.9577752	test: 0.9578138	best: 0.9578191 (444)	total: 23.3s	remaining: 18s
500:	learn: 0.9578163	test: 0.9578246	best: 0.9578388 (476)	total: 25.7s	remaining: 15.3s
550:	learn: 0.9578854

CatBoostClassifier(class_weights=[1.8, 1.0], depth=8, eval_metric='F1', iterations=800, learning_rate=0.06, loss_function='Logloss', od_type='Iter', od_wait=50, random_seed=42, verbose=50)

In [4]:
val_attack_scores = binary_model.predict_proba(x_val)[:, 1]
candidates = []
for threshold in np.linspace(0.50, 0.999, 500):
    pred = (val_attack_scores >= threshold).astype(np.int64)
    fp = int(((pred == 1) & (y_binary_val == 0)).sum())
    tn = int(((pred == 0) & (y_binary_val == 0)).sum())
    tp = int(((pred == 1) & (y_binary_val == 1)).sum())
    fn = int(((pred == 0) & (y_binary_val == 1)).sum())
    fpr = fp / (fp + tn) if fp + tn else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    precision = tp / (tp + fp) if tp + fp else 0.0
    if fpr <= TARGET_MAX_FPR and recall >= MIN_ATTACK_RECALL:
        candidates.append((threshold, fpr, recall, precision))

if candidates:
    BINARY_ATTACK_THRESHOLD, val_fpr, val_recall, val_precision = max(candidates, key=lambda item: item[2])
else:
    # 조건을 만족하지 못하면 오탐 제약을 우선한다.
    rows = []
    for threshold in np.linspace(0.50, 0.999, 500):
        pred = (val_attack_scores >= threshold).astype(np.int64)
        fp = int(((pred == 1) & (y_binary_val == 0)).sum())
        tn = int(((pred == 0) & (y_binary_val == 0)).sum())
        tp = int(((pred == 1) & (y_binary_val == 1)).sum())
        fn = int(((pred == 0) & (y_binary_val == 1)).sum())
        fpr = fp / (fp + tn) if fp + tn else 0.0
        recall = tp / (tp + fn) if tp + fn else 0.0
        precision = tp / (tp + fp) if tp + fp else 0.0
        rows.append((threshold, fpr, recall, precision))
    BINARY_ATTACK_THRESHOLD, val_fpr, val_recall, val_precision = min(rows, key=lambda item: (abs(item[1] - TARGET_MAX_FPR), -item[2]))

BINARY_ATTACK_THRESHOLD, val_fpr, val_recall, val_precision

(np.float64(0.5), 0.00042, 0.9205854291868711, 0.9993273398962181)

In [5]:
attack_classes = np.array(sorted(attack_idx_to_label), dtype=np.int64)
attack_weights = compute_class_weight(class_weight='balanced', classes=attack_classes, y=y_attack_train)
attack_weights = np.minimum(attack_weights, 50.0)  # SQL Injection 초소수 클래스 폭주 방지

attack_model = CatBoostClassifier(
    loss_function='MultiClass',
    eval_metric='TotalF1',
    iterations=800,
    learning_rate=0.08,
    depth=8,
    class_weights=attack_weights.tolist(),
    random_seed=RANDOM_STATE,
    od_type='Iter',
    od_wait=50,
    verbose=50,
)
attack_model.fit(x_attack_train, y_attack_train, eval_set=(x_attack_val, y_attack_val), use_best_model=True)
dict(zip(attack_names, attack_weights.round(4)))

0:	learn: 0.9125130	test: 0.9117479	best: 0.9117479 (0)	total: 146ms	remaining: 1m 56s
50:	learn: 0.9185549	test: 0.9183113	best: 0.9183113 (50)	total: 6.47s	remaining: 1m 34s
100:	learn: 0.9186882	test: 0.9185572	best: 0.9185574 (89)	total: 12.6s	remaining: 1m 27s
150:	learn: 0.9187278	test: 0.9185920	best: 0.9185920 (146)	total: 18.4s	remaining: 1m 19s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.9185920256
bestIteration = 146

Shrink model to first 147 iterations.


{'Bot': np.float64(7.2031),
 'Brute Force': np.float64(0.5648),
 'DDoS': np.float64(0.5648),
 'DoS': np.float64(0.5648),
 'Infiltration': np.float64(1.8199),
 'SQL Injection': np.float64(50.0)}

In [6]:
test_attack_scores = binary_model.predict_proba(x_test)[:, 1]
binary_pred = (test_attack_scores >= BINARY_ATTACK_THRESHOLD).astype(np.int64)
print('=== Binary Report ===')
print(classification_report(y_binary_test, binary_pred, target_names=['Benign', 'Attack'], digits=4, zero_division=0))
print(confusion_matrix(y_binary_test, binary_pred, labels=[0, 1]))

attack_pred = attack_model.predict(x_attack_test).astype(np.int64).reshape(-1)
print('\n=== Attack-Type Report (oracle attack-only input) ===')
print(classification_report(y_attack_test, attack_pred, labels=attack_classes, target_names=attack_names, digits=4, zero_division=0))
print(confusion_matrix(y_attack_test, attack_pred, labels=attack_classes))

=== Binary Report ===
              precision    recall  f1-score   support

      Benign     0.9484    0.9995    0.9733    150000
      Attack     0.9992    0.9198    0.9579    101669

    accuracy                         0.9673    251669
   macro avg     0.9738    0.9597    0.9656    251669
weighted avg     0.9690    0.9673    0.9671    251669

[[149926     74]
 [  8150  93519]]

=== Attack-Type Report (oracle attack-only input) ===
               precision    recall  f1-score   support

          Bot     1.0000    1.0000    1.0000      2353
  Brute Force     0.7183    0.9996    0.8359     30000
         DDoS     0.9998    0.9995    0.9997     30000
          DoS     1.0000    0.6084    0.7566     30000
 Infiltration     0.9998    0.9999    0.9998      9311
SQL Injection     0.3125    1.0000    0.4762         5

     accuracy                         0.8842    101669
    macro avg     0.8384    0.9346    0.8447    101669
 weighted avg     0.9168    0.8842    0.8796    101669

[[ 2353 

In [7]:
binary_path = MODEL_DIR / BINARY_MODEL_NAME
attack_path = MODEL_DIR / ATTACK_MODEL_NAME
binary_model.save_model(binary_path)
attack_model.save_model(attack_path)

binary_report = classification_report(
    y_binary_test,
    binary_pred,
    target_names=['Benign', 'Attack'],
    digits=4,
    zero_division=0,
    output_dict=True,
)
attack_report = classification_report(
    y_attack_test,
    attack_pred,
    labels=attack_classes,
    target_names=attack_names,
    digits=4,
    zero_division=0,
    output_dict=True,
)
meta = {
    'version': 'netflow_v2_twostage_catboost',
    'binary_model': str(binary_path),
    'attack_model': str(attack_path),
    'processed_dir': str(PROCESSED_DIR),
    'binary_attack_threshold': float(BINARY_ATTACK_THRESHOLD),
    'target_max_fpr': TARGET_MAX_FPR,
    'min_attack_recall': MIN_ATTACK_RECALL,
    'attack_class_names': attack_names,
    'attack_label_mapping': attack_label_mapping,
    'binary_report': binary_report,
    'binary_confusion_matrix': confusion_matrix(y_binary_test, binary_pred, labels=[0, 1]).tolist(),
    'attack_report': attack_report,
    'attack_confusion_matrix': confusion_matrix(y_attack_test, attack_pred, labels=attack_classes).tolist(),
}
meta_path = MODEL_DIR / 'model_meta_netflow_v2_twostage.json'
meta_path.write_text(json.dumps(meta, ensure_ascii=False, indent=2), encoding='utf-8')
binary_path, attack_path, meta_path

(PosixPath('/Users/minseok/Desktop/codex/프로젝트/DeepFence/training/artifacts/models/best_model_netflow_v2_binary_catboost.cbm'),
 PosixPath('/Users/minseok/Desktop/codex/프로젝트/DeepFence/training/artifacts/models/best_model_netflow_v2_attack_catboost.cbm'),
 PosixPath('/Users/minseok/Desktop/codex/프로젝트/DeepFence/training/artifacts/models/model_meta_netflow_v2_twostage.json'))